In [ ]:
from dotenv import load_dotenv
load_dotenv()

In [ ]:
import os
import operator

import asyncio
from tavily import TavilyClient, AsyncTavilyClient

from typing import Literal,Annotated
from pydantic import BaseModel

from langchain_groq import ChatGroq
from langchain_core.messages  import HumanMessage, SystemMessage, AIMessage, ToolMessage
from langchain_core.tools  import tool
from langchain_core.prompts import ChatPromptTemplate
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import  RecursiveCharacterTextSplitter
from langchain_google_genai import ChatGoogleGenerativeAI,GoogleGenerativeAIEmbeddings
from langchain_chroma import Chroma
from langchain_core.documents import Document
from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import MessagesState
from langgraph.prebuilt import ToolNode

In [ ]:
#load PDF 

loader = PyPDFLoader("./documents/evs_oil_price_shock.pdf")
raw_docs = loader.load()
print(f"Loaded {len(raw_docs)} pages")

In [ ]:
# split documents into chunks
chunks = RecursiveCharacterTextSplitter(chunk_size =900, chunk_overlap =150).split_documents(raw_docs)
print(f"Number of chunks after splitting  = {len(chunks)} chunks")

In [ ]:
#initializing the embedding model
embeddings =  GoogleGenerativeAIEmbeddings( model = "gemini-embedding-001",# or 'models/gemini-embedding-exp-03-07'
    task_type="retrieval_document"# Options: retrieval_query, retrieval_document, semantic_similarity, etc.
)

In [ ]:
# in-memory only; re-running this cell re-embeds from scratch
vector_store = Chroma(collection_name = "rag_eval_retry",embedding_function = embeddings)
vector_store.add_documents(documents = chunks)
print("Vector store is ready")

In [ ]:
#initializing LLM
llm = ChatGroq(model="openai/gpt-oss-120b")
agent_llm = ChatGroq(model="openai/gpt-oss-120b")
#llm = ChatGoogleGenerativeAI(model = "gemini-2.5-flash")# route_question and generate
#agent_llm = ChatGoogleGenerativeAI(model = "gemini-2.5-flash") # agent node; tools bound later

In [ ]:
#Graph State
class AgenticRAGState(MessagesState):
    query : str
    retrieved_docs : Annotated[list[Document] ,operator.add]
    context : Annotated[str,operator.add]
    generation : str
    needs_retrieval : bool
    retrieval_count: int       # incremented after each tool-call round
    max_retrieval_steps: int   # cap on tool-call rounds before forcing generate

In [ ]:
# structured output schema for the routing decision
class RouteDecision(BaseModel):
    needs_retrieval : bool

In [ ]:
#tool
@tool(response_format = "content_and_artifact")
def vector_store_search(query:str , k:int= 3):
    """
    Search the vector store for relevant document passages.
    Adjust k (default 3) to retrieve more or fewer passages.
    """
    docs = vector_store.similarity_search(query, k = k)
    context = "\n\n## Vector Store Results\n\n" + "\n\n".join(d.page_content for d in docs)
    return  context, docs

In [ ]:
@tool(response_format = "content_and_artifact")
def web_search(query : str, max_results:int =3):
    """
    Search the web for current or real-time information.
    Adjust max_results (default 3) to control how many results are returned.
    """
    client = TavilyClient(api_key=os.environ["TAVILY_API_KEY"])
    response = client.search(query,max_results = max_results)
    docs = [
        Document(
            page_content = r["content"],
            metadata={"source": r["url"], "title": r.get("title", "")},
        )
        for r in response["results"]
    ]
    content = "\n\n## Web Search Results\n\n" + "\n\n".join(d.page_content for d in docs)
    return content, docs

In [ ]:
tools = [vector_store_search,web_search]

agent_llm_with_tools = agent_llm.bind_tools(tools)

In [ ]:
tool_node = ToolNode(tools)

In [ ]:
# route_question node — classifies whether the query needs document retrieval
def route_question(state:AgenticRAGState) -> dict:
    prompt_template = ChatPromptTemplate.from_messages([
        ("system","Classify whether the following question requires retrieving information from a specialized document, or can be answered from your own general knowledge."),
        ("human","{query}"),
    ])
    chain = prompt_template | llm.with_structured_output(RouteDecision)
    decision = chain.invoke({"query" : state["query"]})
    return {"needs_retrieval" : decision.needs_retrieval}

In [ ]:
AGENT_SYSTEM_PROMPT = (
    "You are a retrieval agent with access to two tools:\n\n"
    "1. vector_store_search — use this for questions that can be answered from the internal document: "
    "a technical report titled 'Will EVs Dampen the Oil Price Shock?' covering EV adoption trajectories, "
    "oil demand displacement scenarios, fleet turnover dynamics, battery cost trends, OPEC+ supply behavior, "
    "and energy price volatility projections through 2050. "
    "Use this tool whenever the query references the report, its findings, its projections, or any topic "
    "that would plausibly appear in a domain-specific EV/oil-market research document. "
    "You may increase k beyond the default if broader coverage of the document is needed.\n\n"
    "2. web_search — use this for current or real-time information not covered by the document, such as "
    "recent market data, news, or statistics from 2024 onward. "
    "Always rephrase the query into a concise, keyword-optimized web search string before calling this tool.\n\n"
    "You may call one tool, both tools, or no tool depending on what the query requires. "
    "When both document knowledge and current data are relevant, call both tools."
)
# agent node — decides which tool(s) to call based on the query
def agent(state:AgenticRAGState) -> dict:
    messages = state["messages"]
    if not messages:
        messages = [SystemMessage(content=AGENT_SYSTEM_PROMPT),
                    HumanMessage(content=state["query"])]
    response =agent_llm_with_tools.invoke(messages)
    return {"messages": [response]}

In [ ]:
# collect_tool_output node — drains ToolMessages into state fields
def collect_tool_output(state: AgenticRAGState) -> dict:
    tool_messages = [m for m in state["messages"] if isinstance(m, ToolMessage)]
    all_docs, context_parts = [], []
    for msg in tool_messages:
        context_parts.append(msg.content)
        all_docs.extend(msg.artifact)

    return {
        "context": "\n\n".join(context_parts),
        "retrieved_docs": all_docs,
    }

In [ ]:
# generates a response using the retrieved context
def generate(state: AgenticRAGState) -> dict:
    query = state["query"]
    messages = state["messages"]
    context = state.get("context") or ""
    # Clean message history for Gemini: Remove dangling ToolMessages 
    # since their content is already safely packed into the 'context' variable
    cleaned_messages = [msg for msg in messages if not isinstance(msg, ToolMessage)]
    if context:
        prompt_template = ChatPromptTemplate.from_messages([
        ("system", "Answer the question using only the context below.\n\nContext:\n{context}"),
        ("human", "{query}"),
        ])

        response = (prompt_template | llm).invoke({"context": context, "query": query})
    else:
        prompt_template = ChatPromptTemplate.from_messages([
            ("system", "Answer the following question from your general knowledge."),
            ("human", "{query}"),
        ])
        response = (prompt_template | llm).invoke({"query": query})
    return {"generation": response.content}

In [ ]:
# check_retrieval_limit node — increments the retrieval counter after each tool round
def check_retrieval_limit(state: AgenticRAGState) -> dict:
    count = state.get("retrieval_count", 0)
    return {"retrieval_count": count + 1}

In [ ]:
def route_after_classification(state: AgenticRAGState) -> Literal["agent", "generate"]:
    return "agent" if state["needs_retrieval"] else "generate"

def route_after_agent(state: AgenticRAGState) -> Literal["tools", "collect_tool_output"]:
    return "tools" if state["messages"][-1].tool_calls else "collect_tool_output"

def route_after_limit_check(state: AgenticRAGState) -> Literal["agent", "collect_tool_output"]:
    return "collect_tool_output" if state["retrieval_count"] >= state["max_retrieval_steps"] else "agent"

In [ ]:
#build graph
builder = StateGraph(AgenticRAGState)

builder.add_node("route_question",route_question)
builder.add_node("agent",agent)
builder.add_node("tools",tool_node)
builder.add_node("check_retrieval_limit",check_retrieval_limit)
builder.add_node("collect_tool_output",collect_tool_output)
builder.add_node("generate",generate)

builder.add_edge(START,"route_question")
builder.add_conditional_edges("route_question",route_after_classification)
builder.add_conditional_edges("agent",route_after_agent)
builder.add_edge("tools", "check_retrieval_limit")
builder.add_conditional_edges("check_retrieval_limit", route_after_limit_check)
builder.add_edge("collect_tool_output","generate")
builder.add_edge("generate", END)

graph = builder.compile()
graph

In [ ]:
# Case 1: vector store only — domain-specific question about the PDF
query_vs = "According to the report, how will the adoption of electric vehicles impact oil demand?"

result_vs = graph.invoke({"query": query_vs, "messages": [], "retrieved_docs": [], "context": ""})


In [ ]:
print("=== CASE 1: Vector Store Only ===")
print(f"needs_retrieval : {result_vs['needs_retrieval']}")
print(f"retrieved_docs  : {len(result_vs.get('retrieved_docs') or [])} docs")
print(f"context preview : {result_vs.get('context', '')[:120]}...")
print(f"\nGeneration:\n{result_vs['generation']}")

In [ ]:
print(result_vs["context"])

In [ ]:
# Case 2: web search only — current information not in the PDF
query_web = "What is the current temperature in New Delhi?"

result_web = graph.invoke({"query": query_web, "messages": [], "retrieved_docs": [], "context": ""})

In [ ]:
print("=== CASE 2: Web Search Only ===")
print(f"needs_retrieval : {result_web['needs_retrieval']}")
print(f"retrieved_docs  : {len(result_web.get('retrieved_docs') or [])} docs")
print(f"context preview : {result_web.get('context', '')[:120]}...")
print(f"\nGeneration:\n{result_web['generation']}")

In [ ]:
# Case 3: both tools — requires document context AND current web data
query_both = "Compare what the EV report projects for 2030 oil demand and also tell me about the weather in mumbai"

result_both = graph.invoke({"query": query_both, "messages": [], "retrieved_docs": [], "context": ""})

In [ ]:
print("=== CASE 3a: Both Tools ===")
print(f"needs_retrieval : {result_both['needs_retrieval']}")
print(f"retrieved_docs  : {len(result_both.get('retrieved_docs') or [])} docs")
print(f"context preview : {result_both.get('context', '')[:120]}...")
print(f"\nGeneration:\n{result_both['generation']}")

In [ ]:
print(result_both["context"])

In [ ]:
# Case 3b: multi-round retrieval — query designed to trigger iterative tool calls
# The agent is expected to: (1) search the vector store for report projections,
# then (2) search the web for current data, then (3) search again to fill any gaps.
query_multi = (
    "Step 1: Look up what the EV report projects for oil demand displacement by 2030 and 2050. "
    "Step 2: Search for the latest 2024-2025 EV sales data and battery cost trends. "
    "Step 3: Search for any recent analyst forecasts that revise those oil demand projections upward or downward. "
    "Synthesize all three into a coherent answer."
)

result_multi = graph.invoke({"query": query_multi, "messages": [], "retrieved_docs": [], "context": "", "retrieval_count": 0, "max_retrieval_steps": 5})

In [ ]:
print("=== CASE 3b: Multi-Round Retrieval ===")
print(f"needs_retrieval  : {result_multi['needs_retrieval']}")
print(f"retrieval_count  : {result_multi['retrieval_count']}")
print(f"retrieved_docs   : {len(result_multi.get('retrieved_docs') or [])} docs")
print(f"context preview  : {result_multi.get('context', '')[:120]}...")
print(f"\nGeneration:\n{result_multi['generation']}")

In [ ]:
# Case 4: no retrieval — answerable from general knowledge
query_none = "Explain the difference between kinetic energy and potential energy."

result_none = graph.invoke({"query": query_none, "messages": [], "retrieved_docs": [], "context": ""})

In [ ]:
print("=== CASE 4: No Retrieval ===")
print(f"needs_retrieval : {result_none['needs_retrieval']}")
print(f"retrieved_docs  : {len(result_none.get('retrieved_docs') or [])} docs")
print(f"context preview : {result_none.get('context', '') or '(none)'}")
print(f"\nGeneration:\n{result_none['generation']}")

In [ ]:
result_both["messages"][0].tool_calls